In [ ]:
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
from imblearn import FunctionSampler
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

DATASET = Path("Datasets/Processed/Dataset Dupilumab Baseline.xlsx - Anagrafica.csv")


def load_dataset():
    for root in (Path.cwd(), *Path.cwd().parents):
        if (root / DATASET).is_file():
            return pd.read_csv(root / DATASET)
    raise FileNotFoundError(DATASET)

class MultiColumnBinarizer(BaseEstimator, TransformerMixin):
    # BaseEstimator: provides parameter handling for sklearn Pipelines and GridSearch
    # TransformerMixin: automatically implements the fit_transform() method

    def __init__(self, min_frequency):
        self.min_frequency = min_frequency
        self.binarizers_ = {}
        self.col_names_ = []
        self.frequent_classes_ = {}


    def _parse_series(self, series):
        series = series.fillna("").astype(str) # Missing values become 0
        parsed = []
        for row in series:
            if not row.strip():
                parsed.append([])
            else:
                items = [
                    item.strip().lower()
                    for item in row.split(",")
                    if item.strip()
                ]
                parsed.append(list(set(items)))
        return parsed

    def _map_rare_to_other(self, lists, freq_set):
        # Maps infrequent or unseen categories into 'other'
        return [list({item if item in freq_set else "other" for item in sublist}) for sublist in lists]

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        self.col_names_ = list(X_df.columns)
        self.binarizers_ = {}
        self.frequent_classes_ = {}

        for col in self.col_names_:
            lists = self._parse_series(X_df[col])
            
            # Count occurrences in the training set and keep classes >= min_frequency
            all_items = [item for sublist in lists for item in sublist]
            counts = pd.Series(all_items).value_counts()
            freq_set = set(counts[counts >= self.min_frequency].index)
            self.frequent_classes_[col] = freq_set

            # Map rare classes to 'other' before fitting MultiLabelBinarizer
            mapped_lists = self._map_rare_to_other(lists, freq_set)
            all_classes = sorted(list(freq_set | {"other"}))

            mlb = MultiLabelBinarizer(classes=all_classes, sparse_output=False)
            mlb.fit(mapped_lists)
            self.binarizers_[col] = mlb

        return self
   

    def transform(self, X):
        X_df = pd.DataFrame(X, columns=self.col_names_)
        transformed_parts = []
        
        for col in self.col_names_:
            lists = self._parse_series(X_df[col])
            mapped_lists = self._map_rare_to_other(lists, self.frequent_classes_[col])
            transformed_parts.append(self.binarizers_[col].transform(mapped_lists))
            
        return np.hstack(transformed_parts)

    def get_feature_names_out(self, input_features=None):
        feature_names = []
        for col in self.col_names_:
            prefix = col.lower().replace(" ", "_")
            classes = self.binarizers_[col].classes_
            feature_names.extend([f"{prefix}_{c.replace(' ', '_')}" for c in classes])
        return np.array(feature_names)

def smotenc_resample(X, y, fattore, random_state):
    if fattore == 1:
        return X, y
    prefixes = tuple(col.lower().replace(" ", "_") + "_" for col in rf_multi_categorical_cols)
    cat_idx = [i for i, col in enumerate(X.columns)
               if col in rf_categorical_cols or col in rf_ordinal_cols or col.startswith(prefixes)]
    return SMOTENC(
        categorical_features=cat_idx,
        sampling_strategy={label: round(n * fattore) for label, n in Counter(y).items()},
        k_neighbors=5,
        random_state=random_state,
    ).fit_resample(X, y)


In [ ]:
# RandomForest | set C
raw_rf = load_dataset()

rf_continuous_cols = [
    "Peso (kg)",
    "Altezza (cm)",
    "Neutrofili",
    "Linfociti",
    "Monociti",
    "Eosinofili",
    "Basofili",
    "IgE totali",
]

rf_discrete_cols = [
    'Età',
    'Num Comorbidità',
    'Numero di pregressi interventi ESS',
    'N. allergeni positivi',
    # 'Giorni al mese di utilizzo spray',
    'ACT',
    'NPS',
    'Physical Symptoms',
    'Health and QoL',
    'Nasal Symptoms',
    'Otologic Symptoms',
    'Sleep Symptoms',
    'Emotional Symptoms',
    'NCS',
    'Dolore facciale',
    'Ostruzione nasale',
    'Rinorrea anteriore',
    'Rinorrea posteriore',
    'Starnuti',
    'QoL',
]

rf_categorical_cols = [
    "Categoria Professione",
    "Sesso",
    "Biologico precedente (0 no, 1 sì)",
    "Familiarità Atopia",
    "OCS nell'ultimo anno",
    "Intolleranza ai FANS",
]

rf_multi_categorical_cols = [
    "Comorbidità",
    "Quali allergeni positivi",
]

rf_ordinal_cols = [
    "Fumo Baseline (0=no, 1=ex, 2=si)",
]

rf_numeric = rf_continuous_cols + rf_discrete_cols + rf_ordinal_cols
rf_features = rf_numeric + rf_categorical_cols + rf_multi_categorical_cols
X_rf = raw_rf.loc[:, rf_features].copy()
y_rf = raw_rf["TARGET"].astype(int)

for col in rf_numeric:
    values = X_rf[col].astype("string").str.strip()
    european = values.str.contains(",", regex=False, na=False)
    values = values.mask(european, values.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
    X_rf[col] = pd.to_numeric(values, errors="coerce").astype(float)

rf_pre = ColumnTransformer([
    ("num_cont", make_pipeline(SimpleImputer(strategy="mean"), StandardScaler()), rf_continuous_cols),
    ("num_discr", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), rf_discrete_cols),
    ("cat", SimpleImputer(strategy="most_frequent"), rf_categorical_cols),
    ("multi_cat", MultiColumnBinarizer(min_frequency=2), rf_multi_categorical_cols),
    ("ord", make_pipeline(SimpleImputer(strategy="most_frequent"), StandardScaler()), rf_ordinal_cols),
], verbose_feature_names_out=False).set_output(transform="pandas")

rf_post = ColumnTransformer([
    ("cat", OneHotEncoder(sparse_output=False, drop="if_binary", handle_unknown="ignore"), rf_categorical_cols),
], remainder="passthrough")

# Configurazione fissa dalla griglia di Final_Choices_Test (non selezionata per fold).
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=1,
    max_features="log2",
    random_state=42,
    n_jobs=-1,
)
rf_smotenc = FunctionSampler(
    func=smotenc_resample,
    validate=False,
    kw_args={"fattore": 1.5, "random_state": 42},
)
rf_pipeline = ImbPipeline([
    ("pre", rf_pre),
    ("smotenc", rf_smotenc),
    ("post", rf_post),
    ("model", rf_model),
])
rf_pipeline.fit(X_rf, y_rf)


In [ ]:
# VALIDAZIONE | parametri fissati nella cella precedente
# Stesso schema esterno di Final_Choices_Test: 5 fold x 2 ripetizioni.
# Ogni fold rif? preprocessing e SMOTENC soltanto sul training.
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import RepeatedStratifiedKFold

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42)
validation_rows = []
for split_num, (train_idx, test_idx) in enumerate(cv.split(X_rf, y_rf)):
    fold_pipeline = clone(rf_pipeline)
    fold_pipeline.set_params(
        smotenc__kw_args={"fattore": rf_smotenc.kw_args["fattore"], "random_state": 42 + split_num}
    )
    fold_pipeline.fit(X_rf.iloc[train_idx], y_rf.iloc[train_idx])
    predicted = fold_pipeline.predict(X_rf.iloc[test_idx])
    actual = y_rf.iloc[test_idx]
    validation_rows.append({
        "ripetizione": split_num // 5 + 1,
        "fold": split_num % 5 + 1,
        "f1_macro": f1_score(actual, predicted, average="macro"),
        "accuracy": accuracy_score(actual, predicted),
        "precision": precision_score(actual, predicted, zero_division=0),
        "recall": recall_score(actual, predicted, zero_division=0),
    })

validation_df = pd.DataFrame(validation_rows)
validation_summary = validation_df[
    ["f1_macro", "accuracy", "precision", "recall"]
].agg(["mean", "std"]).T
print("Metriche per fold")
display(validation_df.round(4))
print("Media e deviazione standard")
display(validation_summary.round(4))


In [ ]:
# EXPLAINABILITY | setup comune
import matplotlib.pyplot as plt
import shap

from sklearn.inspection import permutation_importance, PartialDependenceDisplay

best_pipeline = rf_pipeline
pre = best_pipeline.named_steps["pre"]
post = best_pipeline.named_steps["post"]
model = best_pipeline.named_steps["model"]

X_explain = X_rf.copy()
y_explain = y_rf.copy()
X_pre = pre.transform(X_explain)
X_transformed = post.transform(X_pre)
feature_names = post.get_feature_names_out()
X_transformed_df = pd.DataFrame(
    X_transformed, columns=feature_names, index=X_explain.index
).astype(float)

print("Original features:", X_explain.shape[1])
print("Transformed features:", X_transformed_df.shape[1])
print("Samples:", len(X_explain))
print("Classes:", model.classes_)

In [ ]:
# PERMUTATION FEATURE IMPORTANCE | feature originali
# Calcolata sui dati del fit: descrive il modello, non stima la generalizzazione.
permutation = permutation_importance(
    best_pipeline, X_explain, y_explain,
    scoring="f1_macro", n_repeats=20, random_state=42, n_jobs=-1,
)
perm_df = pd.DataFrame({
    "Feature": X_explain.columns,
    "Importance": permutation.importances_mean,
    "Std": permutation.importances_std,
}).sort_values("Importance", ascending=False)
display(perm_df)

top_perm = perm_df.head(15).sort_values("Importance")
plt.figure(figsize=(9, 6))
plt.barh(top_perm["Feature"], top_perm["Importance"],
         xerr=top_perm["Std"], capsize=3)
plt.xlabel("Riduzione di F1 macro dopo permutazione")
plt.title("RandomForest | Permutation importance")
plt.tight_layout()
plt.show()

In [ ]:
# SHAP | spiegazione globale della probabilit? classe 1
# TreeExplainer produce due classi: selezioniamo la classe 1.
background = shap.sample(
    X_transformed_df, min(30, len(X_transformed_df)), random_state=42
)
X_shap = shap.sample(
    X_transformed_df, min(150, len(X_transformed_df)), random_state=123
)
explainer = shap.TreeExplainer(model)
raw_sv = explainer(X_shap.to_numpy())
sv = shap.Explanation(
    values=raw_sv.values[:, :, 1],
    base_values=raw_sv.base_values[:, 1],
    data=X_shap.to_numpy(),
    feature_names=list(feature_names),
)
shap.plots.bar(sv, max_display=15)
shap.plots.beeswarm(sv, max_display=15)

In [ ]:
# SHAP | spiegazione locale, eseguibile anche senza la cella SHAP globale
patient_idx = 0
patient_row = X_transformed_df.iloc[[patient_idx]]
patient = patient_row.to_numpy(dtype=float)
background_local = shap.sample(
    X_transformed_df, min(30, len(X_transformed_df)), random_state=42
)
local_explainer = shap.TreeExplainer(model)
raw_local = local_explainer(patient)
local_sv = shap.Explanation(
    values=raw_local.values[0, :, 1],
    base_values=raw_local.base_values[0, 1],
    data=patient[0],
    feature_names=list(feature_names),
)
print("Patient index:", patient_row.index[0])
print("True label:", y_explain.loc[patient_row.index[0]])
print("Predicted class:", model.predict(patient)[0])
print("Probability class 1:", model.predict_proba(patient)[0, 1])
shap.plots.waterfall(local_sv, max_display=15)

In [ ]:
# LIME | spiegazione locale della probabilit? classe 1
import lime.lime_tabular

def lime_predict_fn(X):
    return model.predict_proba(np.asarray(X, dtype=float))

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_transformed_df.to_numpy(),
    feature_names=list(feature_names),
    class_names=["No eosinofilia", "Eosinofilia"],
    mode="classification",
    discretize_continuous=True,
    random_state=42,
)
patient_idx = 43
exp = lime_explainer.explain_instance(
    data_row=X_transformed_df.iloc[patient_idx].to_numpy(),
    predict_fn=lime_predict_fn,
    num_features=12,
    num_samples=3000,
)
# Il grafico standard di LIME mostra soglie nello spazio trasformato.
# Qui ogni barra riporta anche il valore della feature originale del paziente.
from textwrap import fill

patient_id = X_transformed_df.index[patient_idx]
original_row = X_explain.loc[patient_id]
original_columns = sorted(X_explain.columns, key=len, reverse=True)

def lime_feature_label(feature_index):
    encoded_name = str(feature_names[feature_index])
    name = encoded_name.split("__", 1)[-1]
    for original_name in original_columns:
        multi_prefix = original_name.lower().replace(" ", "_") + "_"
        if name == original_name:
            indicator = None
        elif name.startswith(original_name + "_"):
            indicator = name[len(original_name) + 1:]
        elif original_name in rf_multi_categorical_cols and name.startswith(multi_prefix):
            indicator = name[len(multi_prefix):]
        else:
            continue
        value = original_row[original_name]
        value_text = "mancante" if pd.isna(value) else str(value)
        label = f"{original_name} = {value_text}"
        if indicator is not None:
            encoded_value = X_transformed_df.iloc[patient_idx, feature_index]
            label += f" | {indicator}: {encoded_value:g}"
        return fill(label, width=55)
    return fill(f"{encoded_name} = {X_transformed_df.iloc[patient_idx, feature_index]:g}", width=55)

contributions = exp.as_map()[1][:12]
indices, weights = zip(*reversed(contributions))
labels = [lime_feature_label(i) for i in indices]
fig, ax = plt.subplots(figsize=(12, max(5, 0.72 * len(labels))))
ax.barh(range(len(weights)), weights,
        color=["#2a9d6f" if value >= 0 else "#d9534f" for value in weights])
ax.set_yticks(range(len(labels)), labels)
ax.axvline(0, color="#555", linewidth=0.8)
ax.set_xlabel("Contributo LIME alla classe Eosinofilia")
ax.set_title(f"LIME | Paziente {patient_id} | valori originali")
fig.subplots_adjust(left=0.49, right=0.98, top=0.94, bottom=0.09)
plt.show()


In [ ]:
# PARTIAL DEPENDENCE E ICE | feature numeriche trasformate
numeric_features = rf_continuous_cols + rf_discrete_cols + rf_ordinal_cols
top_numeric = [col for col in perm_df["Feature"] if col in numeric_features][:4]
features_to_plot = []
for col in top_numeric:
    matches = [i for i, name in enumerate(feature_names)
               if name == col or name.endswith("__" + col)]
    if len(matches) == 1:
        features_to_plot.append(matches[0])
print("Features selected:", [feature_names[i] for i in features_to_plot])
if not features_to_plot:
    raise ValueError("Nessuna feature numerica trovata per PDP/ICE.")
fig, ax = plt.subplots(figsize=(12, 8), layout="constrained")
PartialDependenceDisplay.from_estimator(
    estimator=model,
    X=X_transformed_df.to_numpy(),
    features=features_to_plot,
    feature_names=list(feature_names),
    kind="both",
    response_method="predict_proba",
    subsample=50,
    n_cols=2,
    grid_resolution=30,
    random_state=42,
    ax=ax,
    ice_lines_kw={"color": "#5bc0de", "alpha": 0.25, "linewidth": 0.8},
    pd_line_kw={"color": "#d9534f", "linewidth": 2.2},
)
fig.suptitle("RandomForest | Partial dependence e ICE")
plt.show()

In [ ]:
# SHAP | focus sulle comorbidit?
# Eseguire dopo la cella SHAP globale: sv e X_shap si riferiscono agli stessi pazienti.
# I valori SHAP sono nelle unit? dell'output spiegato dal modello, non sono effetti causali.
comorbidity_col = next(col for col in X_explain.columns if col.lower().startswith("comorbidit"))
comorbidity_prefix = comorbidity_col.lower().replace(" ", "_") + "_"
comorbidity_indices = [
    i for i, name in enumerate(feature_names)
    if str(name).split("__", 1)[-1].startswith(comorbidity_prefix)
]
if not comorbidity_indices:
    raise ValueError("Nessun indicatore di comorbidit? tra le feature trasformate.")

shap_matrix = np.asarray(sv.values)
if shap_matrix.ndim != 2 or shap_matrix.shape != X_shap.shape:
    raise ValueError("Esegui prima la cella SHAP globale per questo notebook.")

comorbidity_rows = []
for i in comorbidity_indices:
    category = str(feature_names[i]).split("__", 1)[-1][len(comorbidity_prefix):]
    present = X_shap.iloc[:, i].to_numpy() > 0
    contribution = shap_matrix[:, i]
    comorbidity_rows.append({
        "Comorbidit?": category,
        "Pazienti presenti": int(present.sum()),
        "SHAP medio assoluto": float(np.abs(contribution).mean()),
        "SHAP medio se presente": float(contribution[present].mean()) if present.any() else np.nan,
        "SHAP medio se assente": float(contribution[~present].mean()) if (~present).any() else np.nan,
    })
comorbidity_shap_df = pd.DataFrame(comorbidity_rows).sort_values(
    "SHAP medio assoluto", ascending=False
)
display(comorbidity_shap_df.round(4))

plot_df = comorbidity_shap_df.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, max(4, 0.42 * len(plot_df))))
ax.barh(plot_df["Comorbidit?"], plot_df["SHAP medio assoluto"])
ax.set_xlabel("Contributo SHAP medio assoluto")
ax.set_title("Comorbidit? | importanza globale")
fig.tight_layout()
plt.show()

# Dettaglio locale del primo paziente spiegato in X_shap.
patient_position = 0
patient_id = X_shap.index[patient_position]
local_rows = []
for i in comorbidity_indices:
    category = str(feature_names[i]).split("__", 1)[-1][len(comorbidity_prefix):]
    local_rows.append({
        "Comorbidit?": category,
        "Presente": bool(X_shap.iloc[patient_position, i] > 0),
        "SHAP": float(shap_matrix[patient_position, i]),
    })
local_comorbidity_df = pd.DataFrame(local_rows).sort_values(
    "SHAP", key=lambda s: s.abs(), ascending=False
)
print("Paziente:", patient_id)
print("Comorbidit? originali:", X_explain.loc[patient_id, comorbidity_col])
display(local_comorbidity_df.round(4))

local_plot = local_comorbidity_df.head(12).iloc[::-1]
labels = [
    f"{row['Comorbidit?']} ({'presente' if row['Presente'] else 'assente'})"
    for _, row in local_plot.iterrows()
]
fig, ax = plt.subplots(figsize=(9, max(4, 0.42 * len(labels))))
ax.barh(labels, local_plot["SHAP"],
        color=["#2a9d6f" if value >= 0 else "#d9534f" for value in local_plot["SHAP"]])
ax.axvline(0, color="#555", linewidth=0.8)
ax.set_xlabel("Contributo SHAP alla previsione")
ax.set_title(f"Comorbidit? | paziente {patient_id}")
fig.tight_layout()
plt.show()
